# Demo 1 · MinHash 近重复检测

In [ ]:
%pip install -q pymilvus==3.0.2 requests

## 1. 下载数据

In [ ]:
import json, requests
from collections import defaultdict

URL = ("https://raw.githubusercontent.com/zc277584121/milvus3-demos/main/"
       "demos/minhash-near-duplicate/data/original-short-stories-r1/documents.json")
docs = requests.get(URL, timeout=30).json()["documents"]
print(len(docs), "篇")

## 2. 建集合：签名交给函数自动生成

**你不用自己算 MinHash。** `content` 开启分词，`minhash_signature` 列由 `Function(MINHASH)` 在插入时自动填充。签名维度 = num_hashes × 32bit = 8192。

In [ ]:
from pymilvus import MilvusClient, DataType, Function, FunctionType

client = MilvusClient(uri="http://127.0.0.1:19530")
COLL = "minhash_demo"
if client.has_collection(COLL):
    client.drop_collection(COLL)

schema = client.create_schema(auto_id=False, enable_dynamic_field=False)
schema.add_field("doc_id", DataType.VARCHAR, is_primary=True, max_length=64)
schema.add_field("title", DataType.VARCHAR, max_length=256)
schema.add_field("cluster", DataType.VARCHAR, max_length=64)
schema.add_field("variant", DataType.VARCHAR, max_length=32)
schema.add_field("content", DataType.VARCHAR, max_length=8192, enable_analyzer=True)
schema.add_field("minhash_signature", DataType.BINARY_VECTOR, dim=8192)
schema.add_function(Function(
    name="minhash_fn", function_type=FunctionType.MINHASH,
    input_field_names=["content"], output_field_names=["minhash_signature"],
    params={"num_hashes": 256, "shingle_size": 1},
))
client.create_collection(COLL, schema=schema)

idx = client.prepare_index_params()
idx.add_index(field_name="minhash_signature", index_type="MINHASH_LSH",
              metric_type="MHJACCARD",
              params={"mh_element_bit_width": 32, "with_raw_data": True})
client.create_index(COLL, index_params=idx)

client.insert(COLL, [{"doc_id": d["id"], "title": d["title"], "cluster": d["cluster"],
                      "variant": d["variant"], "content": d["content"]} for d in docs])
client.flush(COLL)
client.load_collection(COLL)
print(client.get_collection_stats(COLL))

## 3. 搜索：拿一篇"轻编辑稿"找原文

In [ ]:
light = next(d for d in docs if d["variant"] == "light-edits")
res = client.search(COLL, data=[light["content"]], anns_field="minhash_signature",
                    search_params={"metric_type": "MHJACCARD",
                                   "params": {"mh_search_with_jaccard": True, "refine_k": 10}},
                    limit=5, output_fields=["title", "variant"])
print("查询:", light["title"], "(light-edits)")
for h in res[0]:
    print(f'{h["distance"]:.4f}  {h["entity"]["variant"]:18s} {h["entity"]["title"]}')

## 4. 变体矩阵：7 个原稿 × 各自变体的实测分数

In [ ]:
for cluster in sorted(by_cluster):
    orig = next((d for d in docs if d["cluster"] == cluster and d["variant"] == "original"), None)
    if not orig:
        continue
    res = client.search(COLL, data=[orig["content"]], anns_field="minhash_signature",
                        search_params={"metric_type": "MHJACCARD",
                                       "params": {"mh_search_with_jaccard": True, "refine_k": 10}},
                        limit=10, output_fields=["cluster", "variant"])
    rows = sorted([(h["entity"]["variant"], round(h["distance"], 3))
                   for h in res[0] if h["entity"]["cluster"] == cluster], key=lambda x: -x[1])
    print(f"{cluster:26s}", rows)

## 5. 阈值

看全库"每篇文档最像的另一篇"分数分布——变体和无关文之间没有灰区。

In [ ]:
import statistics
scores = []
for d in docs:
    res = client.search(COLL, data=[d["content"]], anns_field="minhash_signature",
                        search_params={"metric_type": "MHJACCARD",
                                       "params": {"mh_search_with_jaccard": True, "refine_k": 10}},
                        limit=2, output_fields=["doc_id"])
    for h in res[0]:
        if h["entity"]["doc_id"] != d["id"]:
            scores.append(float(h["distance"]))
            break
print("n =", len(scores), "| median =", round(statistics.median(scores), 3),
      "| min =", round(min(scores), 3), "| max =", round(max(scores), 3))
print(">0.5 :", sum(s > 0.5 for s in scores))
print("0.2~0.5:", sum(0.2 < s <= 0.5 for s in scores))
print("<=0.2:", sum(s <= 0.2 for s in scores))


1. `distance` 是 Jaccard 相似度的估计值，越大越像，可以直接当相似度用；
2. `shingle_size` 用默认值 1：本 demo 里变体相似度在 0.72~1.0，无关文本在 0.2 以下，区分度已经足够；
3. `refine_k=10`：LSH 粗筛出候选后，用原始签名再精算 top-k。


## 清理

In [ ]:
client.drop_collection("minhash_demo")
print("done")